# Donor Upsell Prediction
Predicting which donors are likely to increase their giving level.

## 1. Problem Framing
Donor upsell (upgrade) prediction identifies existing donors who are most likely to respond positively to an ask for a higher gift level. This maximises fundraising yield by focusing major-gift solicitations on donors with high upgrade propensity.

**Target variable:** `upgraded` — binary (1 = increased year-over-year gift by ≥ 20%, 0 = flat or declined).

**Success metric:** Precision ≥ 0.65 for class 1 (minimise wasted outreach), ROC-AUC ≥ 0.78.

**Stakeholders:** Major gifts officer, development team.

## 2. Data Acquisition & Preparation

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
n = 700

df = pd.DataFrame({
    'donor_id': range(1, n + 1),
    'tenure_years': np.random.uniform(0.5, 15, size=n),
    'last_year_gift': np.random.lognormal(4.5, 1.1, size=n),
    'lifetime_value': np.random.lognormal(6.0, 1.2, size=n),
    'num_years_giving': np.random.randint(1, 16, size=n),
    'avg_gift_trend': np.random.uniform(-0.5, 0.5, size=n),  # YoY pct change
    'engagement_score': np.random.randint(0, 101, size=n),
    'volunteer_hours': np.random.exponential(5, size=n).clip(0, 200),
    'event_invites_accepted': np.random.randint(0, 6, size=n),
    'wealth_rating': np.random.choice(['Low', 'Medium', 'High', 'Very High'], size=n,
                                       p=[0.3, 0.4, 0.2, 0.1]),
    'recent_major_life_event': np.random.randint(0, 2, size=n),
})

wealth_map = {'Low': 0, 'Medium': 1, 'High': 2, 'Very High': 3}
df['wealth_encoded'] = df['wealth_rating'].map(wealth_map)

upgrade_prob = (
    0.1
    + 0.10 * (df['avg_gift_trend'] > 0).astype(float)
    + 0.002 * df['engagement_score']
    + 0.05 * df['wealth_encoded']
    + 0.03 * df['event_invites_accepted']
    + 0.08 * df['recent_major_life_event']
).clip(0.05, 0.90)
df['upgraded'] = np.random.binomial(1, upgrade_prob)

print(df.shape)
print(df['upgraded'].value_counts())
df.head()

In [ ]:
feature_cols = [
    'tenure_years', 'last_year_gift', 'lifetime_value', 'num_years_giving',
    'avg_gift_trend', 'engagement_score', 'volunteer_hours',
    'event_invites_accepted', 'wealth_encoded', 'recent_major_life_event'
]

X = df[feature_cols]
y = df['upgraded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 3. Exploration

In [ ]:
import matplotlib.pyplot as plt

# Upgrade rate by wealth rating
wr_rate = df.groupby('wealth_rating')['upgraded'].mean().reindex(['Low', 'Medium', 'High', 'Very High'])
wr_rate.plot(kind='bar', color='goldenrod', figsize=(6, 3))
plt.title('Upgrade Rate by Wealth Rating')
plt.ylabel('Proportion Upgraded')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Engagement score distribution by upgrade class
fig, ax = plt.subplots(figsize=(7, 3))
df[df['upgraded'] == 1]['engagement_score'].hist(bins=20, alpha=0.6, ax=ax, label='Upgraded', color='green')
df[df['upgraded'] == 0]['engagement_score'].hist(bins=20, alpha=0.6, ax=ax, label='Not upgraded', color='gray')
ax.set_title('Engagement Score by Upgrade Status')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Modeling
### 4a. Predictive Model — XGBoost / Gradient Boosting Classifier
### 4b. Causal / Explanatory Model — SHAP (TreeExplainer)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# --- 4a: Predictive model ---
gbc = GradientBoostingClassifier(
    n_estimators=200, max_depth=3, learning_rate=0.05,
    subsample=0.8, random_state=42
)
gbc.fit(X_train, y_train)

y_pred_proba = gbc.predict_proba(X_test)[:, 1]
y_pred = gbc.predict(X_test)
print("Gradient Boosting training complete.")

In [ ]:
# --- 4b: Causal / Explanatory model using SHAP ---
try:
    import shap
    explainer = shap.TreeExplainer(gbc)
    shap_values = explainer.shap_values(X_test)

    sv = shap_values[1] if isinstance(shap_values, list) else shap_values

    shap.summary_plot(sv, X_test, feature_names=feature_cols, show=False)
    plt.title('SHAP Summary — Donor Upsell (Class 1 = Upgraded)')
    plt.tight_layout()
    plt.show()

    # Single-donor waterfall
    idx = 0
    shap.waterfall_plot(
        shap.Explanation(
            values=sv[idx],
            base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list)
                        else explainer.expected_value,
            data=X_test.iloc[idx],
            feature_names=feature_cols
        ),
        show=False
    )
    plt.title('SHAP Waterfall — Single Donor')
    plt.tight_layout()
    plt.show()
except ImportError:
    print("shap not installed — skipping. Install with: pip install shap")

## 5. Evaluation

In [ ]:
from sklearn.metrics import (
    classification_report, roc_auc_score,
    RocCurveDisplay, ConfusionMatrixDisplay,
    precision_recall_curve, average_precision_score
)

print("ROC-AUC:", round(roc_auc_score(y_test, y_pred_proba), 4))
print("Average Precision:", round(average_precision_score(y_test, y_pred_proba), 4))
print()
print(classification_report(y_test, y_pred, target_names=['Not Upgraded', 'Upgraded']))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
RocCurveDisplay.from_predictions(y_test, y_pred_proba, ax=axes[0], name='GBC')
axes[0].set_title('ROC Curve — Donor Upsell')

prec, rec, thresh = precision_recall_curve(y_test, y_pred_proba)
axes[1].plot(rec, prec, color='purple')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=['Not Upgraded', 'Upgraded'], ax=axes[2]
)
axes[2].set_title('Confusion Matrix')

plt.tight_layout()
plt.show()

## 6. Feature Selection

In [ ]:
importances = pd.Series(gbc.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(8, 4))
importances.plot(kind='bar', color='goldenrod')
plt.title('Feature Importances — Donor Upsell')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()

# Cumulative importance threshold
cum_importance = importances.cumsum()
selected_features = cum_importance[cum_importance <= 0.90].index.tolist()
if not selected_features:
    selected_features = importances.head(6).index.tolist()
print("Selected features (90% cumulative importance):", selected_features)

## 7. Deployment

In [ ]:
import pickle, os

os.makedirs('models', exist_ok=True)

X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

gbc_final = GradientBoostingClassifier(
    n_estimators=200, max_depth=3, learning_rate=0.05,
    subsample=0.8, random_state=42
)
gbc_final.fit(X_train_sel, y_train)

with open('models/donor_upsell_model.pkl', 'wb') as f:
    pickle.dump({'model': gbc_final, 'features': selected_features}, f)

print("Model saved to models/donor_upsell_model.pkl")

# Prioritised outreach list
scored = X_test_sel.copy()
scored['upgrade_probability'] = gbc_final.predict_proba(scored)[:, 1]
scored['donor_id'] = X_test.index
outreach_list = scored.sort_values('upgrade_probability', ascending=False).head(20)
print("Top 20 donors to solicit for upgrade:")
print(outreach_list[['donor_id', 'upgrade_probability']].to_string(index=False))